### caching transformers
It is used when pipelines are expensive to fit (large datasets or complex preprocessing). In scikit-learn we cache transformers using joblib Memory so that repeated runs reuse fitted transformers instead of recomputing them.

#### Import libraries

In [31]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer

from joblib import Memory

#### Load dataset

In [32]:
df= pd.read_csv(r'E:\MLP\data\raw\GA_2_dataset.csv')
df.head()

,PlayerID,Age,Gender,Location,GameGenre,PlayTimeHours,InGamePurchases,GameDifficulty,SessionsPerWeek,AvgSessionDurationMinutes,PlayerLevel,AchievementsUnlocked,EngagementLevel
0,35900,37.0,Male,Other,Strategy,23.929404,NaN,Hard,3,124,99,18,Medium
1,27085,25.0,Male,NaN,Action,22.755168,1.0,Easy,14,84,84,12,Medium
2,39595,24.0,Female,Europe,Simulation,19.505292,0.0,Hard,3,172,9,18,Medium
3,37440,26.0,Female,Europe,RPG,11.009645,NaN,NaN,3,83,36,43,Low
4,22882,17.0,Female,USA,RPG,0.581039,1.0,Medium,5,163,9,24,Medium


#### Drop PlayerID column from the dataset

In [33]:
df=df.drop("PlayerID",axis=1)

#### Separate feature and target 

In [34]:
X = df.drop("EngagementLevel", axis=1)
y = df["EngagementLevel"]   


#### Encode target verivable


In [35]:
target_encoder = OrdinalEncoder()

y = target_encoder.fit_transform(y.values.reshape(-1,1))
y = y.ravel()

#### Train test split

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Replace unknown values


In [37]:
X_train["Age"] = pd.to_numeric(X_train["Age"], errors="coerce")
X_test["Age"] = pd.to_numeric(X_test["Age"], errors="coerce")

X_train["Location"].replace("Unknown", "Other", inplace=True)
X_test["Location"].replace("Unknown", "Other", inplace=True)

X_train["InGamePurchases"].replace("Unknown", 0, inplace=True)
X_test["InGamePurchases"].replace("Unknown", 0, inplace=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_5240\2034321924.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  X_train["Location"].replace("Unknown", "Other", inplace=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_5240\2034321924.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained a

6252    0.0
4684    0.0
1731    0.0
4742    0.0
4521    1.0
       ... 
6412    1.0
8285    0.0
7853    0.0
1095    0.0
6929    1.0
Name: InGamePurchases, Length: 2000, dtype: float64

#### Define feature group

In [38]:
num_features = [
    "Age",
    "PlayTimeHours",
    "InGamePurchases",
    "SessionsPerWeek",
    "AvgSessionDurationMinutes",
    "PlayerLevel",
    "AchievementsUnlocked"
]

ordinal_features = ["GameDifficulty"]

nominal_features = ["Gender", "Location", "GameGenre"]

#### Create a cache directory

In [39]:
memory = Memory(location="cacha_dir", verbose=0)


#### Numerical pipeline

In [42]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
memory=memory

#### categorical pipeline

In [43]:
ordinal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(categories=[["Easy","Medium","Hard"]]))
    ],
    memory=memory
)

#### Nominal pipeline

In [44]:
nominal_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ],
    memory=memory
)

#### Column transformer

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_features),
        ("ord", ordinal_pipeline, ordinal_features),
        ("nom", nominal_pipeline, nominal_features)
    ]
)

#### Fit and Transform train data

In [49]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [50]:
X_train_processed = pd.DataFrame(X_train_processed)
X_test_processed = pd.DataFrame(X_test_processed)

X_train_processed.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,0.237505,0.544185,-0.471296,-0.776439,-0.395780,1.066241,0.574413,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.972126,-1.306672,-0.471296,-0.429136,-0.457519,-0.544018,-1.370718,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,1.706747,-1.664028,-0.471296,1.133727,1.394638,-0.193962,-1.509656,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,-1.336682,-0.848820,-0.471296,-0.602788,-0.169405,-0.824063,0.574413,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.657289,1.391964,-0.471296,0.786424,0.427401,-0.368990,-0.745497,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


In [51]:
np.round(X_test_processed[:5].sum().sum(), 2)

np.float64(6.86)

#### Without cache
Pipeline run → recompute all transformers
#### With cache
First run → transformers fitted and stored
Next run → reused from cache (faster)